# GTSRB Traffic-Sign Recognition

This notebook downloads the complete GTSRB classification dataset, trains a custom CNN and MobileNetV2, evaluates both, and predicts a sample image. Enable a GPU runtime for full training.

In [ ]:
!pip -q install tensorflow==2.20.0 opencv-python-headless==4.12.0.88 numpy==2.2.6 pandas==2.3.2 matplotlib==3.10.6 scikit-learn==1.7.2 Pillow==11.3.0

In [ ]:
# Upload or clone this repository into /content/mait-mini before running this cell.
from pathlib import Path
PROJECT_DIR = '/content/mait-mini'
assert Path(PROJECT_DIR, 'scripts/download_dataset.py').exists(), 'Place the repository at /content/mait-mini first.'
import sys, subprocess
sys.path.insert(0, PROJECT_DIR + '/src')
subprocess.run([sys.executable, PROJECT_DIR + '/scripts/download_dataset.py'], cwd=PROJECT_DIR, check=True)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from traffic_sign_recognition.data import load_labeled_data
train, test = load_labeled_data(Path(PROJECT_DIR) / 'data/raw/gtsrb')
print(f'train={len(train):,}, test={len(test):,}, classes={train.label.nunique()}')
train.label.value_counts().sort_index().plot.bar(figsize=(16, 4), title='GTSRB training images per class'); plt.show()

In [ ]:
import os
os.chdir(PROJECT_DIR)
!python -m traffic_sign_recognition.train --model custom_cnn --epochs 20 --batch-size 128 --output-dir artifacts/models/custom_cnn
!python -m traffic_sign_recognition.train --model mobilenet_v2 --epochs 15 --batch-size 128 --output-dir artifacts/models/mobilenet_v2

In [ ]:
from IPython.display import display, Image
for model_name in ('custom_cnn', 'mobilenet_v2'):
    print(model_name)
    !python -m traffic_sign_recognition.evaluate --model-path artifacts/models/{model_name}/model.keras
    display(Image(filename=f'artifacts/models/{model_name}/evaluation/confusion_matrix.png', width=650))

In [ ]:
# Predict any uploaded or sample sign image.
sample = test.iloc[0].path
!python -m traffic_sign_recognition.predict --model-path artifacts/models/mobilenet_v2/model.keras --image {sample}